# 🧠 MURE LLM Fine-Tuning Notebook (Production-Ready)
**Gemma-2-2B Fine-Tuning with Unsloth + LoRA**

### လုပ်ဆောင်ရမည့်အဆင့်များ:
1. Google Colab မှာ **GPU Runtime** ရွေးပါ (Runtime → Change runtime type → T4 GPU)
2. **Runtime → Run All** နှိပ်ပါ
3. Google Drive mount လုပ်ရန် popup ပေါ်လာသောအခါ Allow နှိပ်ပါ
4. `svo cc brain` folder ထဲသို့ `mure_finetune_2M.jsonl` ဖိုင် upload လုပ်ပါ

**Expected Training Time:** ~2-4 hours on T4 GPU (Free Colab)

In [ ]:
# ============================================================
# CELL 1: Check GPU + Install Dependencies
# ============================================================
import torch
print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('⚠️ GPU မတွေ့ပါ! Runtime → Change runtime type → GPU ကို ရွေးပါ!')


In [ ]:
# ============================================================
# CELL 2: Install Unsloth + Dependencies (one-time setup)
# ============================================================
import torch
major_version, minor_version = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0,0)
# Install unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
if major_version >= 8:
    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
else:
    !pip install --no-deps "xformers<0.0.27" "trl==0.8.6" peft accelerate bitsandbytes
print('✅ Dependencies installed!')


In [ ]:
# ============================================================
# CELL 3: Mount Google Drive + Load Dataset
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random

DRIVE_FOLDER = '/content/drive/MyDrive/svo cc brain'
os.makedirs(DRIVE_FOLDER, exist_ok=True)

# Dataset priority: 2M JSONL > 50k sample > rules.json fallback
DATASET_FILE = '/content/dataset.jsonl'

jsonl_2m   = os.path.join(DRIVE_FOLDER, 'mure_finetune_2M.jsonl')
jsonl_50k  = os.path.join(DRIVE_FOLDER, 'mure_finetune_50k.jsonl')
any_jsonl  = [f for f in os.listdir(DRIVE_FOLDER) if f.endswith('.jsonl')] if os.path.exists(DRIVE_FOLDER) else []
rules_json = os.path.join(DRIVE_FOLDER, 'rules.json')

if os.path.exists(jsonl_2m):
    print(f'✅ 2M dataset found! Using {jsonl_2m}')
    !cp "{jsonl_2m}" "{DATASET_FILE}"
elif os.path.exists(jsonl_50k):
    print(f'✅ 50k sample dataset found! Using {jsonl_50k}')
    !cp "{jsonl_50k}" "{DATASET_FILE}"
elif any_jsonl:
    src = os.path.join(DRIVE_FOLDER, any_jsonl[0])
    print(f'✅ Found JSONL: {src}')
    !cp "{src}" "{DATASET_FILE}"
elif os.path.exists(rules_json):
    print('⚠️  No JSONL found. Converting rules.json → JSONL...')
    TEMPLATES = [
        ('What happens when {cause}?', 'When {cause}, {effect}. (Confidence: {s:.2f})'),
        ('What is the effect of {cause}?', 'The effect is {effect}. (Strength: {s:.2f})'),
        ('If {cause}, what follows?', 'If {cause}, then {effect} will follow.'),
        ('What leads to {effect}?', '{cause} is known to lead to {effect}.'),
        ('What could cause {effect}?', 'A possible cause of {effect} is {cause}.'),
        ('{cause} ဆိုတာ ဘာကို ဖြစ်စေသလဲ?', '{cause} ကြောင့် {effect} ဖြစ်ပေါ်ပါသည်။'),
    ]
    with open(rules_json, 'r', encoding='utf-8') as f:
        data = json.load(f)
    rules = data.get('causalMemory', []) if isinstance(data, dict) else data
    written = 0
    with open(DATASET_FILE, 'w', encoding='utf-8') as out:
        # Expand each rule with all templates
        for rule in rules:
            cause = str(rule.get('cause', '')).strip()
            effect = str(rule.get('effect', '')).strip()
            s = float(rule.get('strength', rule.get('confidence', 0.8)))
            if not cause or not effect:
                continue
            for tmpl_q, tmpl_a in TEMPLATES:
                try:
                    instruction = tmpl_q.format(cause=cause, effect=effect, s=s)
                    output = tmpl_a.format(cause=cause, effect=effect, s=s)
                    out.write(json.dumps({'instruction': instruction, 'input': '', 'output': output}, ensure_ascii=False) + '\n')
                    written += 1
                except:
                    pass
    print(f'✅ Converted rules.json → {written} JSONL entries')
else:
    raise FileNotFoundError(f'⚠️  Dataset မတွေ့ပါ! {DRIVE_FOLDER} ထဲသို့ mure_finetune_2M.jsonl ဖိုင်ကို upload လုပ်ပါ!')

# Verify dataset
line_count = sum(1 for _ in open(DATASET_FILE, encoding='utf-8'))
size_mb = os.path.getsize(DATASET_FILE) / (1024*1024)
sample = json.loads(open(DATASET_FILE).readline())
print(f'\n📊 Dataset Stats:')
print(f'   Entries : {line_count:,}')
print(f'   Size    : {size_mb:.1f} MB')
print(f'   Sample  : {str(sample)[: 120]}...')


In [ ]:
# ============================================================
# CELL 4: Load Base Model (Gemma-2-2B via Unsloth)
# ============================================================
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None           # None = auto-detect (bfloat16 on A100, float16 on T4)
LOAD_IN_4BIT = True    # 4-bit quantization to fit in 15GB VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/gemma-2-2b-it-bnb-4bit',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print('✅ Base model loaded!')

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                     # LoRA rank (16 is balanced)
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,            # Scaling factor (2x rank is standard)
    lora_dropout=0.05,        # Small dropout for regularization
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=True,          # Rank-stabilized LoRA (better convergence)
    loftq_config=None,
)
print('✅ LoRA adapters added!')
model.print_trainable_parameters()


In [ ]:

# Format dataset
def formatting_func(examples):
    texts = []
    for i, _ in enumerate(examples['instruction']):
        texts.append(f"<start_of_turn>user\n{examples['instruction'][i]}<end_of_turn>\n<start_of_turn>model\n{examples['output'][i]}<end_of_turn>")
    return {'text': texts}

from datasets import load_dataset
dataset = load_dataset('json', data_files=DATASET_FILE, split='train')
dataset = dataset.map(formatting_func, batched=True)

%%capture\n# ============================================================\n# CELL 2: Install Unsloth + Dependencies (one-time setup)\n# ============================================================\nimport torch\nif torch.cuda.is_available():\n    major_version, minor_version = torch.cuda.get_device_capability()\nelse:\n    major_version, minor_version = 0, 0\n\n# Install unsloth\n!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"\nif major_version >= 8:\n    # A100, H100 (Ampere or newer)\n    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes\nelse:\n    # T4 (Turing)\n    !pip install --no-deps "xformers<0.0.27" "trl==0.8.6" peft accelerate bitsandbytes\n\n# Helper for bfloat16\ndef is_bfloat16_supported():\n    return torch.cuda.is_available() and torch.cuda.is_bf16_supported()\n\nprint("✅ Dependencies installed!")\ntrainer.train()


In [ ]:
import trl
TRL_VER = tuple(int(x) for x in trl.__version__.split('.')[:2])
if TRL_VER >= (0, 9):
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainingArguments
    sft_cfg = SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=2,
        packing=True,
        output_dir='/content/mure_ckpt',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=1,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=50,
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        seed=3407,
        report_to='none',
    )
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=dataset, args=sft_cfg)
else:
    from trl import SFTTrainer
    from transformers import TrainingArguments
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=2,
        packing=True,
        args=TrainingArguments(
            output_dir='/content/mure_ckpt',
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            num_train_epochs=1,
            learning_rate=2e-4,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=50,
            optim='adamw_8bit',
            weight_decay=0.01,
            lr_scheduler_type='cosine',
            seed=3407,
            report_to='none',
        )
    )


In [ ]:
# ============================================================
# CELL 7: Save LoRA Adapter to Google Drive
# ============================================================
import os

LORA_DIR = os.path.join(DRIVE_FOLDER, 'MURE_Finetuned_LoRA')
print(f'💾 Saving LoRA adapter to {LORA_DIR} ...')
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print('✅ LoRA adapter saved! (Compact - only stores weight differences)')

# Optional: Also save merged full model (larger, ~5GB)
# FULL_DIR = os.path.join(DRIVE_FOLDER, 'MURE_Finetuned_Full')
# model.save_pretrained_merged(FULL_DIR, tokenizer, save_method='merged_16bit')
# print('✅ Full merged model saved!')

# Optional: Save as GGUF for llama.cpp / Ollama
# GGUF_DIR = os.path.join(DRIVE_FOLDER, 'MURE_GGUF')
# model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method='q4_k_m')
# print('✅ GGUF model saved (for llama.cpp / Ollama)!')


In [ ]:
# ============================================================
# CELL 8: Test the Fine-Tuned Model
# ============================================================
FastLanguageModel.for_inference(model)

TEST_QUESTIONS = [
    'What happens when temperature increases?',
    'What is the effect of acid and base reacting?',
    'မိုးသည်းထန်စွာ ရွာသွန်းသောအခါ ဘာဖြစ်သလဲ?',
    'What leads to increased global temperature over time?',
]

for question in TEST_QUESTIONS:
    prompt = ALPACA_PROMPT.format(question, '', '')
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id, 
            **inputs,
            max_new_tokens=128,
            temperature=0.7,
            do_sample=True,
            use_cache=True,
        )
    # Decode only new tokens (not the prompt)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n❓ Q: {question}')
    print(f'💡 A: {response.strip()}')
    print('-' * 60)

print('\n✅ MURE Fine-Tuning Complete! Model is ready for production.')
